# Statistical Significance Tests — Reference Notebook

A structured reference of statistical tests used in quantitative research, organized by use case: regression, classification, and trading strategy evaluation. Each test includes the question it answers, theory, a reusable implementation, and calibrated example commentary.

Each code cell is **self-contained**: imports are repeated per cell so any cell can be copy-pasted independently.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, f1_score, accuracy_score, log_loss,
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from collections import defaultdict
from itertools import combinations

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

---
# Part I — Regression

## 1.0 OLS with standard p-values (baseline)

`regression` · `cross-section` · `time-series` · `coefficient inference` · **Run on: train or train+dev**

**Question:** Which coefficients are significantly different from zero under classical OLS assumptions (homoscedastic, uncorrelated errors)?

**When to use:** First step of any regression analysis. Provides baseline p-values that are valid only if residuals are homoscedastic and uncorrelated. Run diagnostics next (Breusch-Pagan in 1.1, Durbin-Watson in 1.3), then upgrade SEs if needed.

**Applies to:** Any regression — but standard errors are rarely the final answer. This is the starting point, not the conclusion.

**Theory.** OLS minimises the sum of squared residuals. Under Gauss-Markov assumptions (linearity, exogeneity, homoscedasticity, no serial correlation), OLS is BLUE. The variance of the coefficient vector is estimated as s^2 (X'X)^{-1}, where s^2 = RSS / (n - k). Each t-statistic is t_j = beta_hat_j / SE(beta_hat_j), compared to a t(n-k) distribution.

These p-values are conditional on the assumptions holding. If residuals are heteroscedastic or autocorrelated, the SEs are wrong (usually too small), and p-values are anticonservative.

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd

# ── Inputs ──
# ols_df    : pd.DataFrame with target column and feature columns
# formula   : str, Patsy formula, e.g. "price_per_sqm ~ surface_m2 + C(city)"
#             dynamic construction:
#             formula = "y ~ " + " + ".join(num_cols + [f"C({c})" for c in cat_cols])

ols_model = smf.ols(formula=formula, data=ols_df).fit()

# ── Coefficient table with p-values ──
alpha = 0.05
coef_table = pd.DataFrame({
    "coef": ols_model.params,
    "std_err": ols_model.bse,
    "t_stat": ols_model.tvalues,
    "p_value": ols_model.pvalues,
})
coef_table["significant"] = coef_table["p_value"] < alpha
display(coef_table.sort_values("p_value"))

print(f"\nSignificant at {alpha}: {coef_table['significant'].sum()} / {len(coef_table)} coefficients")
print(f"R-squared: {ols_model.rsquared:.4f}")
print(f"F-stat: {ols_model.fvalue:.1f} (p = {ols_model.f_pvalue:.2e})")

# Expected output:
# ──────────────────────────────────────────────────────
#                       coef   std_err   t_stat   p_value  significant
# C(city)[T.Paris]   5304.00     53.21    99.68    0.0000         True
# Intercept          6842.12    112.30    60.93    0.0000         True
# surface_m2          -10.51      1.34    -7.84    0.0000         True
# C(condition)[T.Good] 312.50     41.20     7.59    0.0000         True
# ...
# building_age_years    -1.02      0.51    -2.00    0.0430         True
# heating_pump          42.30     22.10     1.91    0.0560        False
#
# Significant at 0.05: 24 / 27 coefficients
# R-squared: 0.9180
# F-stat: 412.3 (p = 0.00e+00)

**Example commentary:**

> *"Baseline OLS gives R^2 = 0.918 with 27 coefficients. All city dummies and condition variables are highly significant (p < 0.001). `building_age_years` is marginally significant (p = 0.04). Before interpreting these p-values, we need to check homoscedasticity (Breusch-Pagan) and autocorrelation (Durbin-Watson)."*

> *"The F-statistic (412, p ~ 0) confirms the model is jointly significant — but this does not validate individual coefficient p-values if the error structure is wrong."*

> *"Standard OLS shows 8 significant predictors at 5%. However, the residual plots suggest heteroscedasticity — these p-values are provisional until we confirm with Breusch-Pagan."*

## 1.1 Breusch-Pagan / White test (heteroscedasticity diagnostic)

`regression` · `cross-section` · `residual diagnostic` · **Run on: train or train+dev**

**Question:** Is the residual variance constant across observations, or does it depend on the features / predicted values? If it does, standard OLS p-values (1.0) are wrong.

**When to use:** Immediately after fitting OLS. If rejected, switch to HC3 robust SEs (1.2). Also useful to understand *where* the model is less precise.

**Applies to:** Cross-sectional regression, real estate pricing, wage regressions, any tabular regression.

**Theory.** Both tests regress squared residuals e_i^2 on explanatory variables and check whether the resulting R^2 is significant.

**Breusch-Pagan:** regresses e_i^2 on the original regressors X. Test stat: BP = n * R^2_aux, chi-squared(k). Detects linear heteroscedasticity.

**White's test:** regresses e_i^2 on X, X^2, and cross-products. More general (detects nonlinear patterns) but uses more degrees of freedom.

Decision rule: if either rejects, use HC3 robust SEs for all inference.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
import matplotlib.pyplot as plt

# ── Inputs ──
# ols_model   : fitted OLS result from smf.ols(...).fit() (the standard OLS from 1.0)

resid = ols_model.resid
exog = ols_model.model.exog

bp_stat, bp_pval, bp_fstat, bp_fpval = het_breuschpagan(resid, exog)
white_stat, white_pval, white_fstat, white_fpval = het_white(resid, exog)

print(f"Breusch-Pagan:  stat = {bp_stat:.2f}, p = {bp_pval:.4f}")
print(f"White's test:   stat = {white_stat:.2f}, p = {white_pval:.4f}")

if bp_pval < 0.05 or white_pval < 0.05:
    print("\n-> Heteroscedasticity detected. Use HC3 robust SEs (section 1.2).")
else:
    print("\n-> No evidence of heteroscedasticity. Standard OLS SEs are adequate.")

predicted = ols_model.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(predicted, resid, alpha=0.3, s=12)
axes[0].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Predicted value")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs predicted")
axes[1].scatter(predicted, resid ** 2, alpha=0.3, s=12)
axes[1].set_xlabel("Predicted value")
axes[1].set_ylabel("Squared residual")
axes[1].set_title("Squared residuals vs predicted")
plt.tight_layout()
plt.show()

# Expected output:
# Breusch-Pagan:  stat = 187.43, p = 0.0000
# White's test:   stat = 312.87, p = 0.0000
# -> Heteroscedasticity detected. Use HC3 robust SEs (section 1.2).
# [Plot: fan-shaped residuals, wider spread at higher predicted values]

**Example commentary:**

> *"Both Breusch-Pagan (p < 0.001) and White (p < 0.001) strongly reject homoscedasticity. The residual-squared plot confirms a fan shape above predicted 8,000 EUR/sqm. HC3 SEs are mandatory."*

> *"Breusch-Pagan does not reject (p = 0.34), and the residual plot shows no visible pattern. Standard OLS SEs are adequate."*

> *"White rejects (p = 0.008) but Breusch-Pagan does not (p = 0.12). The heteroscedasticity is nonlinear. HC3 handles this; alternatively, a log-transform of the target could stabilise the variance."*

## 1.2 OLS with HC3 robust standard errors

`regression` · `cross-section` · `coefficient inference` · **Run on: train or train+dev**

**Question:** Are the OLS coefficients still significant after correcting for heteroscedasticity?

**When to use:** After Breusch-Pagan / White (1.1) rejects homoscedasticity. HC3 gives valid p-values regardless of the heteroscedasticity pattern. The coefficients don't change — only the SEs and p-values are corrected.

**Applies to:** Any cross-sectional regression where homoscedasticity is violated.

**Theory.** HC estimators replace the classical variance s^2 (X'X)^{-1} with a sandwich estimator using individual squared residuals.

HC3 applies a leverage correction: each e_i^2 is divided by (1 - h_ii)^2, where h_ii is the leverage. This reduces finite-sample bias compared to HC0 (raw e_i^2) or HC1 (n/(n-k) correction).

HC3 corrects for heteroscedasticity but NOT for serial correlation (use Newey-West, 1.4) or within-group clustering (use clustered SEs, 1.5).

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd

# ── Inputs ──
# ols_df    : pd.DataFrame with target and features
# formula   : str, Patsy formula
#             formula = "y ~ " + " + ".join(num_cols + [f"C({c})" for c in cat_cols])

ols_hc3 = smf.ols(formula=formula, data=ols_df).fit(cov_type="HC3")
ols_standard = smf.ols(formula=formula, data=ols_df).fit()

comparison = pd.DataFrame({
    "coef": ols_standard.params,
    "se_standard": ols_standard.bse,
    "se_hc3": ols_hc3.bse,
    "se_inflation": (ols_hc3.bse / ols_standard.bse).round(2),
    "p_standard": ols_standard.pvalues.round(4),
    "p_hc3": ols_hc3.pvalues.round(4),
})
comparison["sig_lost"] = (comparison["p_standard"] < 0.05) & (comparison["p_hc3"] >= 0.05)

print(f"Coefficients losing significance (standard -> HC3): {comparison['sig_lost'].sum()}")
display(comparison.sort_values("p_hc3"))

# Expected output:
# Coefficients losing significance (standard -> HC3): 1
#
#                     coef  se_standard  se_hc3  se_inflation  p_standard  p_hc3  sig_lost
# C(city)[T.Paris]  5304.0       52.1    53.2         1.02      0.0000  0.0000     False
# surface_m2         -10.5        1.3     1.5         1.12      0.0000  0.0000     False
# building_age        -1.0        0.5     0.6         1.18      0.0310  0.0430     False
# heating_pump        42.3       22.1    28.4         1.28      0.0490  0.1380      True

**Example commentary:**

> *"HC3 SEs are 1.05-1.30x larger than standard SEs. One variable loses significance: `heating_type_heat_pump` (p 0.049 to 0.138). The core city and condition effects are unaffected."*

> *"HC3 correction has minimal impact — SE inflation < 1.05x everywhere. Consistent with the Breusch-Pagan non-rejection."*

> *"Three coefficients lose significance under HC3. All are numerical variables with moderate effect sizes. The large categorical effects are robust."*

## 1.3 Durbin-Watson / Ljung-Box (autocorrelation diagnostic)

`time-series regression` · `return prediction` · `residual diagnostic` · **Run on: train or train+dev**

**Question:** Are the regression residuals serially correlated? If yes, standard and HC3 SEs are wrong — use Newey-West (1.4).

**When to use:** After fitting any time-series regression. If autocorrelation is detected, switch to HAC standard errors. Also diagnostic for model misspecification.

**Applies to:** Any time-series regression, return prediction, macro forecasting.

**Theory.**

**Durbin-Watson:** tests lag-1 autocorrelation. DW ~ 2(1 - rho_1). DW near 2 = no autocorrelation. DW < 1.5 = positive autocorrelation. DW > 2.5 = negative.

**Ljung-Box:** joint test at lags 1 through L. Q = T(T+2) * sum(rho_k^2 / (T-k)), chi-squared(L). More powerful for higher-order autocorrelation. Common lags: 5, 10, 20 (daily); 4, 8, 12 (quarterly).

Decision rule: if DW far from 2 or Ljung-Box rejects, use Newey-West (1.4).

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib.pyplot as plt

# ── Inputs ──
# resid     : np.array or pd.Series of OLS residuals (from ols_model.resid)
# lags      : list of lag values to test

lags = [5, 10, 20]

dw_stat = durbin_watson(resid)
print(f"Durbin-Watson: {dw_stat:.3f}  (target: ~2.0, <1.5 or >2.5 = problem)")

lb_result = acorr_ljungbox(resid, lags=lags, return_df=True)
display(lb_result)

if dw_stat < 1.5 or dw_stat > 2.5 or (lb_result["lb_pvalue"] < 0.05).any():
    print("\n-> Autocorrelation detected. Use Newey-West HAC SEs (section 1.4).")
else:
    print("\n-> No evidence of autocorrelation. OLS or HC3 SEs are adequate.")

fig, ax = plt.subplots(figsize=(10, 3))
sm.graphics.tsa.plot_acf(resid, lags=30, ax=ax, alpha=0.05)
ax.set_title("Residual autocorrelation function")
plt.tight_layout()
plt.show()

# Expected output:
# Durbin-Watson: 1.31  (target: ~2.0, <1.5 or >2.5 = problem)
#     lb_stat  lb_pvalue
# 5    42.31     0.0000
# 10   58.74     0.0000
# 20   71.02     0.0000
# -> Autocorrelation detected. Use Newey-West HAC SEs (section 1.4).
# [ACF plot: significant spikes at lags 1-3]

**Example commentary:**

> *"DW = 2.02, Ljung-Box non-significant at all lags (p > 0.30). No serial correlation — OLS SEs are adequate."*

> *"DW = 1.31 flags positive autocorrelation. Ljung-Box confirms: p < 0.001 at lag 5, ACF shows spikes at lags 1-3. Newey-West mandatory."*

> *"DW = 1.85 is borderline. Ljung-Box at lag 5 gives p = 0.07. Not significant, but HAC correction is prudent — the cost when unnecessary is small, the cost of wrong SEs is large."*

## 1.4 Newey-West (HAC) standard errors

`time-series regression` · `factor models` · `trading signals` · **Run on: train or train+dev**

**Question:** Are my time-series regression coefficients significant after correcting for both autocorrelation and heteroscedasticity?

**When to use:** After Durbin-Watson / Ljung-Box (1.3) detects serial correlation. Newey-West is the time-series analogue of HC3.

**Applies to:** Return prediction, macro forecasting, factor model regressions, any OLS on time-series data.

**Theory.** Newey-West extends HC to account for serial correlation. The HAC variance uses a kernel-weighted sum of autocovariance matrices up to lag L:

V_HAC = (X'X)^{-1} S_hat (X'X)^{-1}, where S_hat includes Bartlett kernel weights w_j = 1 - j/(L+1).

Common lag rule: L = floor(T^{1/4}). Too few lags = residual autocorrelation leaks through (anticonservative). Too many = power loss.

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

# ── Inputs ──
# ts_df     : pd.DataFrame with time-series target and features
# formula   : str, Patsy formula
#             formula = "y ~ " + " + ".join(num_cols + [f"C({c})" for c in cat_cols])

T = len(ts_df)
max_lags = int(T ** 0.25)

ols_standard = smf.ols(formula=formula, data=ts_df).fit()
ols_nw = smf.ols(formula=formula, data=ts_df).fit(
    cov_type="HAC", cov_kwds={"maxlags": max_lags},
)

comparison = pd.DataFrame({
    "coef": ols_standard.params,
    "se_ols": ols_standard.bse,
    "se_nw": ols_nw.bse,
    "se_inflation": (ols_nw.bse / ols_standard.bse).round(2),
    "p_ols": ols_standard.pvalues.round(4),
    "p_nw": ols_nw.pvalues.round(4),
})
comparison["sig_lost"] = (comparison["p_ols"] < 0.05) & (comparison["p_nw"] >= 0.05)

print(f"Newey-West lags: {max_lags} (T = {T})")
print(f"Coefficients losing significance: {comparison['sig_lost'].sum()}")
display(comparison)

# Expected output:
# Newey-West lags: 4 (T = 252)
# Coefficients losing significance: 1
#              coef  se_ols  se_nw  se_inflation  p_ols   p_nw  sig_lost
# Intercept   0.003  0.001  0.001         1.10  0.002  0.008     False
# momentum    0.042  0.012  0.018         1.50  0.001  0.022     False
# volatility -0.015  0.007  0.012         1.71  0.035  0.211      True

**Example commentary:**

> *"Newey-West with 4 lags inflates SEs by 1.3-1.8x. Momentum retains significance (p 0.001 to 0.022), but volatility loses it (p 0.035 to 0.211). The autocorrelation was inflating the volatility coefficient's precision."*

> *"HAC correction has negligible impact (SE inflation 1.0-1.05x). Consistent with DW = 2.01. OLS SEs are adequate."*

> *"After HAC correction (8 lags), the macro predictor's t-stat drops from 3.2 to 1.4. Original significance was an artifact of persistent residuals."*

## 1.5 Clustered standard errors

`regression` · `panel data` · `cross-section with groups` · **Run on: train or train+dev**

**Question:** Are my coefficient p-values still valid when residuals are correlated within groups (districts, firms, time periods)?

**When to use:** When observations are not independent within groups. HC3 handles heteroscedasticity but not within-group correlation.

**Applies to:** Panel regressions, cross-sectional regressions with group structure, event studies.

**Theory.** With G clusters, the robust variance estimator sums X_g' e_g e_g' X_g across groups, allowing arbitrary within-cluster correlation.

Key requirement: at least 40-50 clusters for the asymptotic approximation. With fewer clusters (e.g. 10 states), use wild cluster bootstrap instead.

Clustering absorbs within-group correlation but not between-group heteroscedasticity. For two-way clustering (firm + time), use multi-way clustered SEs.

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd

# ── Inputs ──
# ols_df      : pd.DataFrame with target, features, and cluster variable
# formula     : str, Patsy formula
#               formula = "y ~ " + " + ".join(num_cols + [f"C({c})" for c in cat_cols])
# cluster_var : str, column name to cluster on (e.g. "district", "firm_id")

ols_hc3 = smf.ols(formula=formula, data=ols_df).fit(cov_type="HC3")
ols_clustered = smf.ols(formula=formula, data=ols_df).fit(
    cov_type="cluster", cov_kwds={"groups": ols_df[cluster_var]},
)

comparison = pd.DataFrame({
    "coef": ols_hc3.params,
    "p_hc3": ols_hc3.pvalues,
    "p_clustered": ols_clustered.pvalues,
}).round(4)
comparison["sig_lost"] = (comparison["p_hc3"] < 0.05) & (comparison["p_clustered"] >= 0.05)

n_clusters = ols_df[cluster_var].nunique()
print(f"Clusters: {n_clusters} ({cluster_var})")
print(f"Coefficients losing significance (HC3 -> clustered): {comparison['sig_lost'].sum()}")
display(comparison.sort_values("p_clustered"))

# Expected output:
# Clusters: 140 (district)
# Coefficients losing significance (HC3 -> clustered): 0
#                     coef   p_hc3  p_clustered  sig_lost
# C(city)[T.Paris]  5304.0  0.0000       0.0000     False
# surface_m2         -10.5  0.0000       0.0000     False
# building_age        -1.0  0.0430       0.0453     False

**Example commentary:**

> *"Clustering by district (140 groups) does not overturn any HC3 result. `building_age_years` moves from p = 0.043 to p = 0.045 — razor-thin margin. A finer clustering (building, street) could push it past the threshold."*

> *"After clustering by firm (52 clusters), 3 of 12 significant coefficients lose significance. `R&D_intensity` goes from p = 0.02 to p = 0.11."*

> *"With only 7 clusters (cities), the clustered SEs are unreliable. Wild cluster bootstrap would be more appropriate."*

## 1.6 ADF / KPSS (stationarity)

`time-series` · `cointegration` · `pairs trading` · **Run on: train or full sample**

**Question:** Is this time-series stationary, or does it have a unit root?

**When to use:** Before any time-series regression. Non-stationary series produce spurious results. Run both ADF and KPSS — they have opposite null hypotheses, so joint agreement is stronger evidence.

**Applies to:** Price series, macro variables, spread series for pairs trading.

**Theory.** The two tests are complementary:

**ADF:** H0 = unit root (non-stationary). Rejects = evidence of stationarity.

**KPSS:** H0 = stationary. Rejects = evidence of non-stationarity.

Joint reading: ADF rejects + KPSS does not = **stationary**. ADF does not + KPSS rejects = **unit root, difference the series**. Both reject = possibly **trend-stationary, detrend**. Neither rejects = **ambiguous, low power**.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

# ── Inputs ──
# series    : pd.Series or np.array, the time-series to test

adf_stat, adf_pval, adf_lags, adf_nobs, adf_crit, _ = adfuller(series, autolag="AIC")
print(f"ADF: stat = {adf_stat:.4f}, p = {adf_pval:.4f}, lags = {adf_lags}")
print(f"     Critical values: {adf_crit}")

kpss_stat, kpss_pval, kpss_lags, kpss_crit = kpss(series, regression="c", nlags="auto")
print(f"\nKPSS: stat = {kpss_stat:.4f}, p = {kpss_pval:.4f}, lags = {kpss_lags}")
print(f"      Critical values: {kpss_crit}")

adf_reject = adf_pval < 0.05
kpss_reject = kpss_pval < 0.05

if adf_reject and not kpss_reject:
    print("\n-> Both agree: series is stationary.")
elif not adf_reject and kpss_reject:
    print("\n-> Both agree: unit root present. Difference before modelling.")
elif adf_reject and kpss_reject:
    print("\n-> Conflicting: possibly trend-stationary. Try detrending.")
else:
    print("\n-> Ambiguous: neither rejects. Low power or near-unit-root.")

# Expected output (for a random walk):
# ADF: stat = -1.2340, p = 0.6612, lags = 12
#      Critical values: {'1%': -3.44, '5%': -2.87, '10%': -2.57}
# KPSS: stat = 1.8932, p = 0.0100, lags = 14
# -> Both agree: unit root present. Difference before modelling.

**Example commentary:**

> *"ADF rejects (p = 0.002) and KPSS does not (p = 0.10): the spread series is stationary. Pairs trading viable — it mean-reverts."*

> *"ADF fails (p = 0.45) and KPSS rejects (p < 0.01): unit root. Regressing levels on levels would be spurious. First-differencing required."*

> *"Both reject — likely trend-stationary. Detrending (regress on time, use residuals) is more appropriate than first-differencing."*

## 1.7 Bootstrap confidence intervals on regression metrics

`regression` · `model evaluation` · **Run on: holdout / OOS**

**Question:** How precisely do I know this model's MAE / RMSE / R-squared on the holdout?

**When to use:** Any holdout evaluation. The point estimate (MAE = 570) is one number from one sample — the bootstrap gives a CI. For model comparison, bootstrap the metric difference and check if the CI excludes zero.

**Applies to:** Any regression holdout evaluation.

**Theory.** Resample (y_i, y_hat_i) pairs with replacement B times (typically 2,000). Compute the metric on each resample. The 2.5th and 97.5th percentiles give an approximate 95% CI.

This measures **metric estimation uncertainty** (how precisely do we know the MAE?), not prediction uncertainty for a single observation. For model comparison: delta_b = metric_A(b) - metric_B(b); if the CI excludes 0, the difference is significant.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Inputs ──
# y_actual  : np.array of actual holdout values
# y_pred    : np.array of model predictions on holdout
# y_pred_b  : np.array of second model's predictions (optional, for comparison)

B = 2_000
n = len(y_actual)
rng = np.random.default_rng(42)
boot_rows = []

for _ in range(B):
    idx = rng.choice(n, size=n, replace=True)
    a, p = y_actual[idx], y_pred[idx]
    boot_rows.append({
        "mae": mean_absolute_error(a, p),
        "rmse": mean_squared_error(a, p) ** 0.5,
        "r2": r2_score(a, p),
    })

boot_df = pd.DataFrame(boot_rows)
boot_ci = boot_df.quantile([0.025, 0.5, 0.975]).T
boot_ci.columns = ["ci_2.5%", "median", "ci_97.5%"]
display(boot_ci)

# ── Model comparison (uncomment if y_pred_b available) ──
# boot_delta = []
# for _ in range(B):
#     idx = rng.choice(n, size=n, replace=True)
#     a = y_actual[idx]
#     delta = mean_absolute_error(a, y_pred[idx]) - mean_absolute_error(a, y_pred_b[idx])
#     boot_delta.append(delta)
# ci_lo, ci_hi = np.percentile(boot_delta, [2.5, 97.5])
# print(f"MAE difference CI: [{ci_lo:.1f}, {ci_hi:.1f}]")
# print("Significant" if ci_lo > 0 or ci_hi < 0 else "Not significant")

# Expected output:
#       ci_2.5%  median  ci_97.5%
# mae     535.2   569.5     605.1
# rmse    759.3   819.0     879.4
# r2        0.941   0.948     0.954

**Example commentary:**

> *"Bootstrap 95% CI for MAE: [535, 605] (median 570). The R-squared CI [0.941, 0.954] excludes the OLS-only R-squared (0.918), confirming the boosting model adds statistically significant predictive power."*

> *"MAE difference CI (model A - B): [-42, +18]. Includes zero — cannot conclude A outperforms B."*

> *"RMSE CI [759, 879] is wide relative to the point estimate (819) — the holdout is too small for precise RMSE estimation."*

## 1.8 Permutation test — model vs chance (regression)

`regression` · `model validation` · **Run on: holdout / OOS**

**Question:** Is the model significantly better than a random assignment of targets?

**When to use:** Baseline sanity check. Less informative when the model obviously dominates (R-squared = 0.95), but it closes the statistical hygiene loop. More critical when signal is weak.

**Applies to:** Any regression holdout evaluation, especially low R-squared or small n.

**Theory.** Under H0, predictions have no relationship to targets. Shuffle y N times, recompute the metric each time, build a null distribution. p = (number of null metrics <= actual) / N. Distribution-free, no parametric assumptions. Resolution limited by N (500 permutations -> finest p = 0.002).

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error

# ── Inputs ──
# y_actual  : np.array of actual holdout values
# y_pred    : np.array of model predictions on holdout

n_perm = 500
rng = np.random.default_rng(42)
actual_mae = mean_absolute_error(y_actual, y_pred)
null_maes = np.array([
    mean_absolute_error(rng.permutation(y_actual), y_pred)
    for _ in range(n_perm)
])

p_value = (null_maes <= actual_mae).mean()

print(f"Actual MAE:  {actual_mae:.1f}")
print(f"Null mean:   {null_maes.mean():.1f}")
print(f"Null min:    {null_maes.min():.1f}")
print(f"p-value:     {p_value:.4f}" + (f" (< {1/n_perm:.4f})" if p_value == 0 else ""))

# Expected output:
# Actual MAE:  570.0
# Null mean:   3780.2
# Null min:    3565.4
# p-value:     0.0000 (< 0.0020)

**Example commentary:**

> *"Model MAE (570) lower than all 500 null permutations (null mean 3,780). p < 0.002. Genuine signal confirmed."*

> *"Model MAE (12.3) at 4th percentile of null (p = 0.04). Marginally significant — weak signal, proceed with caution."*

> *"p = 0.18 — performance not distinguishable from random. The apparent R-squared of 0.08 is consistent with noise."*

## 1.9 Bonferroni correction (regression context)

`regression` · `feature selection` · `multiple testing` · **Run on: dev or holdout**

**Question:** After testing K coefficients / features / models simultaneously, can I trust any individual p-value?

**When to use:** When you test multiple hypotheses at once. Without correction, the probability of at least one false positive is 1 - (1-alpha)^K, which grows quickly. Bonferroni divides alpha by K to control the family-wise error rate (FWER). Simple and defensible for moderate K (< 30).

**Applies to:** OLS coefficient tables, feature screening, model selection.

**Theory.** Bonferroni sets the per-test threshold at alpha / K, guaranteeing FWER <= alpha regardless of dependence between tests. Conservative when tests are positively correlated (common with correlated features). Example: testing 10 coefficients at alpha = 0.05 gives threshold 0.005.

In [ ]:
import pandas as pd

# ── Inputs ──
# pvalues     : pd.Series of p-values (index = coefficient/feature names)
#               e.g. from ols_model.pvalues
# alpha       : float, desired FWER (default 0.05)

alpha = 0.05
K = len(pvalues)
bonferroni_threshold = alpha / K

result = pd.DataFrame({
    "p_value": pvalues,
    "bonf_threshold": bonferroni_threshold,
    "sig_raw": pvalues < alpha,
    "sig_bonferroni": pvalues < bonferroni_threshold,
})

n_raw = result["sig_raw"].sum()
n_bonf = result["sig_bonferroni"].sum()

print(f"Tests: {K}, Bonferroni threshold: {bonferroni_threshold:.4f}")
print(f"Significant (raw alpha): {n_raw}")
print(f"Significant (Bonferroni): {n_bonf}")
print(f"Lost to correction: {n_raw - n_bonf}")
display(result.sort_values("p_value"))

# Expected output:
# Tests: 27, Bonferroni threshold: 0.0019
# Significant (raw alpha): 24
# Significant (Bonferroni): 21
# Lost to correction: 3
#                        p_value  bonf_threshold  sig_raw  sig_bonferroni
# C(city)[T.Paris]        0.0000          0.0019     True            True
# surface_m2              0.0000          0.0019     True            True
# heating_pump            0.0060          0.0019     True           False
# building_age            0.0430          0.0019     True           False

**Example commentary:**

> *"Of 27 OLS coefficients, 24 significant at raw alpha. After Bonferroni (threshold 0.0019), 21 survive. The 3 lost were the weakest signals. Core location and condition effects are robust."*

> *"10 features tested, 4 raw-significant. Bonferroni threshold 0.005 — only 1 survives. Apparent signal likely inflated."*

> *"All 5 city coefficients pass Bonferroni easily (all p < 1e-10). Multiple testing only matters for the marginal calls."*

## 1.10 Feature importance stability (regression)

`regression` · `feature selection` · `robustness` · **Run on: dev / CV folds**

**Question:** Is this feature consistently important across data subsets, or does its importance fluctuate?

**When to use:** After permutation importance. A feature ranking #1 in one fold and #15 in another is unreliable. Compute importance on bootstrap samples and measure the coefficient of variation (CV).

**Applies to:** Any ML regression, factor screening, feature selection pipelines.

**Theory.** Feature importance depends on the training sample. By computing importance on B bootstraps, you get a distribution per feature. CV = std / mean: CV < 0.5 is stable, CV > 1.0 is unstable. High mean + low CV = reliable. High mean + high CV = regime-dependent.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

# ── Inputs ──
# model           : fitted sklearn estimator (pipeline or model)
# X_dev, y_dev    : dev set features (pd.DataFrame) and target (pd.Series)
# features        : list of feature names

n_bootstrap = 20
rng = np.random.default_rng(42)
importance_records = []

for b in range(n_bootstrap):
    idx = rng.choice(len(X_dev), size=len(X_dev), replace=True)
    X_b, y_b = X_dev.iloc[idx], y_dev.iloc[idx]
    perm = permutation_importance(
        model, X_b, y_b, n_repeats=10, random_state=42 + b,
        scoring="neg_mean_absolute_error",
    )
    for j, feat in enumerate(features):
        importance_records.append({"bootstrap": b, "feature": feat, "importance": perm.importances_mean[j]})

imp_df = pd.DataFrame(importance_records)
imp_summary = (
    imp_df.groupby("feature")["importance"]
    .agg(["mean", "std"])
    .assign(cv=lambda d: d["std"] / d["mean"].clip(lower=1e-6))
    .sort_values("mean", ascending=False)
)
display(imp_summary.head(15).round(4))

print(f"Stable (CV < 0.5): {(imp_summary['cv'] < 0.5).sum()}")
print(f"Unstable (CV >= 1.0): {(imp_summary['cv'] >= 1.0).sum()}")

# Expected output:
#                       mean      std      cv
# city              2598.412   78.230  0.0301
# district_quality   637.100   45.800  0.0719
# surface_m2          87.200   18.400  0.2110
# log_dist_metro       0.120    0.280  2.3333
# Stable (CV < 0.5): 8
# Unstable (CV >= 1.0): 5

**Example commentary:**

> *"`city` has consistently highest importance (mean 2,598, CV 0.03). Rock-solid. `building_age_years` is unstable (mean 8, CV 1.2) — captures signal only in specific data regimes."*

> *"Top-5 features all have CV < 0.3. Below rank 10, CVs exceed 1.0 — noise-level signal."*

> *"`log_distance_metro` has CV 2.4. Extreme instability from collinearity with `distance_metro_m` — keeping both adds no consistent signal."*

---
# Part II — Classification

## 2.1 Binomial test (accuracy vs baseline)

`classification` · `binary outcomes` · `small holdout` · **Run on: holdout**

**Question:** Is the classifier significantly better than the majority-class baseline?

**When to use:** Binary classification, small holdout. Exact — no asymptotics needed. Baseline is the majority-class rate, not 50%. Assumes IID predictions.

**Applies to:** Churn, credit scoring, medical diagnosis, any binary classifier with limited test data.

**Theory.** Under H0, the number of correct predictions k out of n follows Binomial(n, p0) where p0 = baseline accuracy. One-sided p-value: P(X >= k | Binomial(n, p0)). Exact test, no normal approximation needed.

In [ ]:
from scipy import stats
import pandas as pd
import numpy as np

# ── Inputs ──
# y_actual    : np.array of true binary labels (0/1)
# y_pred      : np.array of predicted binary labels (0/1)

n = len(y_actual)
n_correct = (y_actual == y_pred).sum()
accuracy = n_correct / n
baseline = pd.Series(y_actual).value_counts(normalize=True).max()

p_value = stats.binomtest(n_correct, n, p=baseline, alternative="greater").pvalue

print(f"Accuracy:  {accuracy:.4f} ({n_correct}/{n})")
print(f"Baseline:  {baseline:.4f} (majority class)")
print(f"Lift:      {accuracy - baseline:+.4f}")
print(f"p-value:   {p_value:.4f}")

# Expected output:
# Accuracy:  0.7820 (391/500)
# Baseline:  0.6350
# Lift:      +0.1470
# p-value:   0.0000

**Example commentary:**

> *"Accuracy 78.2% vs baseline 63.5% (n=500). Binomial p < 0.001 — significantly better than majority class."*

> *"Accuracy 66.0% vs baseline 64.2% (n=200). p = 0.29 — the 1.8pp lift is not significant."*

> *"With n=50, even 15pp improvement fails significance (p = 0.08). Holdout too small."*

## 2.2 McNemar's test (comparing two classifiers)

`classification` · `model comparison` · `paired test` · **Run on: holdout**

**Question:** Do two classifiers make significantly different errors on the same holdout?

**When to use:** Comparing two classifiers on identical test data. More powerful than comparing accuracies because it uses the paired structure (only considers discordant observations).

**Applies to:** New model vs baseline, A/B testing classifiers.

**Theory.** Build a 2x2 table: n_10 = A right & B wrong, n_01 = A wrong & B right. Under H0, n_10 = n_01. McNemar's statistic (continuity correction): chi2 = (|n_10 - n_01| - 1)^2 / (n_10 + n_01), compared to chi-squared(1). For small discordant counts (< 25), use exact binomial version.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

# ── Inputs ──
# y_actual    : np.array of true labels
# y_pred_a    : np.array of predictions from model A
# y_pred_b    : np.array of predictions from model B

correct_a = (y_actual == y_pred_a)
correct_b = (y_actual == y_pred_b)

n_10 = ((correct_a) & (~correct_b)).sum()
n_01 = ((~correct_a) & (correct_b)).sum()
n_11 = ((correct_a) & (correct_b)).sum()
n_00 = ((~correct_a) & (~correct_b)).sum()

table = np.array([[n_11, n_10], [n_01, n_00]])
result = mcnemar(table, exact=False, correction=True)

print(f"Both right: {n_11}, Both wrong: {n_00}")
print(f"Discordant: A right/B wrong={n_10}, A wrong/B right={n_01}")
print(f"McNemar stat: {result.statistic:.2f}, p = {result.pvalue:.4f}")

# Expected output:
# Both right: 320, Both wrong: 85
# Discordant: A right/B wrong=47, A wrong/B right=23
# McNemar stat: 7.31, p = 0.0069

**Example commentary:**

> *"Discordant: A gets 47 right that B misses, B gets 23 right that A misses. p = 0.007 — A is significantly better."*

> *"Discordant: 31 vs 28. p = 0.78 — no significant difference."*

> *"Only 12 discordant pairs (8 vs 4). p = 0.39. Too few to distinguish the models."*

## 2.3 Bootstrap confidence intervals on classification metrics

`classification` · `model evaluation` · **Run on: holdout / OOS**

**Question:** How precisely do I know this classifier's AUC / F1 / accuracy?

**When to use:** Any holdout classification evaluation. Critical for imbalanced datasets. For model comparison, bootstrap the metric difference.

**Applies to:** Churn, credit scoring, fraud detection, any binary or multiclass evaluation.

**Theory.** Same as regression bootstrap (1.7). Resample (y_i, p_hat_i) pairs with replacement. Skip resamples with single class (AUC undefined). For model comparison: delta_b = AUC_A(b) - AUC_B(b); CI excludes 0 = significant.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

# ── Inputs ──
# y_actual    : np.array of true binary labels (0/1)
# y_prob      : np.array of predicted probabilities (for AUC)
# y_pred      : np.array of predicted binary labels (for F1 / accuracy)

B = 2_000
n = len(y_actual)
rng = np.random.default_rng(42)
boot_rows = []

for _ in range(B):
    idx = rng.choice(n, size=n, replace=True)
    a, p_prob, p_cls = y_actual[idx], y_prob[idx], y_pred[idx]
    if len(np.unique(a)) < 2:
        continue
    boot_rows.append({
        "auc": roc_auc_score(a, p_prob),
        "f1": f1_score(a, p_cls),
        "accuracy": accuracy_score(a, p_cls),
    })

boot_df = pd.DataFrame(boot_rows)
boot_ci = boot_df.quantile([0.025, 0.5, 0.975]).T
boot_ci.columns = ["ci_2.5%", "median", "ci_97.5%"]
display(boot_ci)

# Expected output:
#           ci_2.5%  median  ci_97.5%
# auc         0.812   0.841     0.867
# f1          0.710   0.745     0.780
# accuracy    0.752   0.782     0.812

**Example commentary:**

> *"AUC CI: [0.812, 0.867]. Excludes 0.80 threshold — model is above the minimum acceptable bar."*

> *"AUC CI: [0.63, 0.74]. Includes 0.70 threshold — cannot confirm the model meets the bar."*

> *"AUC difference CI (A - B): [+0.02, +0.08]. Excludes zero — A is significantly better."*

## 2.4 Calibration test (Hosmer-Lemeshow / reliability diagram)

`classification` · `probabilistic models` · `credit risk` · **Run on: holdout**

**Question:** When the model says "70% probability", does the event actually occur ~70% of the time?

**When to use:** Any probabilistic classifier where probabilities drive decisions (pricing, thresholding, risk). High AUC does not imply good calibration.

**Applies to:** Credit scoring (PD), churn probability, insurance pricing, medical risk scores.

**Theory.** Calibration means P(Y=1 | p_hat = p) = p for all p. The reliability diagram bins predictions and plots observed vs predicted frequency. Perfect calibration = diagonal.

Hosmer-Lemeshow: bins into G groups by predicted probability, computes chi-squared comparing observed and expected. Rejection = miscalibration. Sensitive to G, low power with small samples — the diagram is often more informative.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.calibration import calibration_curve

# ── Inputs ──
# y_actual    : np.array of true binary labels (0/1)
# y_prob      : np.array of predicted probabilities

n_bins = 10
fraction_pos, mean_predicted = calibration_curve(y_actual, y_prob, n_bins=n_bins, strategy="uniform")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot([0, 1], [0, 1], "--", color="grey", linewidth=0.8, label="Perfect")
axes[0].plot(mean_predicted, fraction_pos, "o-", markersize=5, label="Model")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Observed frequency")
axes[0].set_title("Reliability diagram")
axes[0].legend(fontsize=9)
axes[1].hist(y_prob, bins=30, edgecolor="white", linewidth=0.3)
axes[1].set_xlabel("Predicted probability")
axes[1].set_title("Prediction distribution")
plt.tight_layout()
plt.show()

bins = pd.qcut(y_prob, q=n_bins, duplicates="drop")
hl_df = pd.DataFrame({"actual": y_actual, "prob": y_prob, "bin": bins})
hl_grouped = hl_df.groupby("bin", observed=False).agg(
    n=("actual", "size"), observed=("actual", "sum"), expected_rate=("prob", "mean"))
hl_grouped["expected"] = hl_grouped["n"] * hl_grouped["expected_rate"]
hl_stat = ((hl_grouped["observed"] - hl_grouped["expected"]) ** 2 /
           (hl_grouped["expected"] * (1 - hl_grouped["expected_rate"])).clip(lower=1e-10)).sum()
hl_pval = 1 - stats.chi2.cdf(hl_stat, df=len(hl_grouped) - 2)

print(f"Hosmer-Lemeshow: stat = {hl_stat:.2f}, p = {hl_pval:.4f}")

# Expected output:
# [Reliability diagram: curve above diagonal at low p, below at high p]
# Hosmer-Lemeshow: stat = 18.42, p = 0.0031

**Example commentary:**

> *"Systematic overconfidence above 0.6. HL rejects (p = 0.003). Model ranks well (AUC 0.84) but probabilities need recalibration (Platt scaling)."*

> *"Reliability curve close to diagonal, HL does not reject (p = 0.42). Logistic regression is well-calibrated out of the box."*

> *"Good calibration in 20-80% range, breaks down in tails: at predicted > 0.9, observed = 0.75. Matters for auto-approval thresholds."*

## 2.5 Log-loss permutation test

`classification` · `probabilistic models` · **Run on: holdout**

**Question:** Is the classifier's probabilistic output significantly better than chance?

**When to use:** When you care about probability quality, not just ranking. Log-loss penalises confident wrong predictions harshly.

**Applies to:** Credit risk (PD), insurance pricing, medical risk prediction.

**Theory.** Log-loss (cross-entropy): L = -(1/n) * sum [y_i * log(p_hat_i) + (1-y_i) * log(1-p_hat_i)]. Lower is better. Baseline: always predict the base rate. Permutation test: shuffle labels, recompute log-loss N times, build null distribution.

In [ ]:
import numpy as np
from sklearn.metrics import log_loss

# ── Inputs ──
# y_actual    : np.array of true binary labels (0/1)
# y_prob      : np.array of predicted probabilities

n_perm = 500
rng = np.random.default_rng(42)
y_prob_clipped = np.clip(y_prob, 1e-7, 1 - 1e-7)

actual_logloss = log_loss(y_actual, y_prob_clipped)
null_loglosses = np.array([
    log_loss(rng.permutation(y_actual), y_prob_clipped) for _ in range(n_perm)
])

p_value = (null_loglosses <= actual_logloss).mean()
base_rate = y_actual.mean()
baseline_logloss = log_loss(y_actual, np.full_like(y_prob, base_rate))

print(f"Model log-loss:    {actual_logloss:.4f}")
print(f"Baseline log-loss: {baseline_logloss:.4f}")
print(f"Null mean:         {null_loglosses.mean():.4f}")
print(f"p-value:           {p_value:.4f}")

# Expected output:
# Model log-loss:    0.4210
# Baseline log-loss: 0.6830
# Null mean:         0.7120
# p-value:           0.0000

**Example commentary:**

> *"Model log-loss 0.42 vs baseline 0.68. p < 0.002. The probabilities are significantly informative."*

> *"Model log-loss 0.64 vs baseline 0.66. p = 0.12. Despite AUC 0.72, the probability estimates add almost no value."*

> *"Log-loss permutation p = 0.03 but calibration rejects (HL p = 0.002). Signal exists but probabilities are miscalibrated. Recalibrate first."*

## 2.6 Bonferroni correction (classification context)

`classification` · `model selection` · `multiple testing` · **Run on: dev or holdout**

**Question:** After evaluating K classifier variants, can I trust the best one?

**When to use:** Selecting the best from K candidates on holdout AUC. The "best of K" has inflated performance.

**Applies to:** Hyperparameter tuning, threshold selection, feature subset comparison.

**Theory.** Same as 1.9. If you compare K classifiers and declare the best "significant" at alpha, the corrected threshold is alpha / K. Apply to bootstrap or permutation p-values from model comparison.

In [ ]:
import pandas as pd

# ── Inputs ──
# model_results : pd.DataFrame with columns: model_name, holdout_auc, p_value
# alpha         : float (default 0.05)

alpha = 0.05
K = len(model_results)
bonf_threshold = alpha / K

model_results = model_results.sort_values("holdout_auc", ascending=False).copy()
model_results["sig_raw"] = model_results["p_value"] < alpha
model_results["sig_bonf"] = model_results["p_value"] < bonf_threshold

print(f"Models: {K}, Bonferroni threshold: {bonf_threshold:.4f}")
print(f"Significant (raw): {model_results['sig_raw'].sum()}")
print(f"Significant (Bonf): {model_results['sig_bonf'].sum()}")
display(model_results)

# Expected output:
# Models: 8, Bonferroni threshold: 0.0063
# Significant (raw): 5
# Significant (Bonf): 3

**Example commentary:**

> *"8 variants tested. Best AUC 0.841 (p = 0.001). Bonferroni threshold 0.006 — top 3 of 5 raw-significant models survive."*

> *"50 combinations tested. Best p = 0.01. Bonferroni threshold 0.001 — not significant after correcting for the search."*

> *"Only 3 models. Bonferroni threshold 0.017 — barely more stringent than raw alpha."*

## 2.7 Feature importance stability (classification)

`classification` · `feature selection` · `robustness` · **Run on: dev / CV folds**

**Question:** Is this feature consistently important for classification across data subsets?

**When to use:** Same as 1.10, with classification metric (AUC drop). Critical for regulated models (credit scoring) where feature stability affects regulatory acceptance.

**Applies to:** Churn, credit scoring, fraud detection.

**Theory.** Same as 1.10. Permutation importance on each bootstrap using `roc_auc` scoring. CV = std/mean per feature measures stability. For regulated models, demonstrating stable feature ranking may be required.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

# ── Inputs ──
# model           : fitted sklearn classifier
# X_dev, y_dev    : dev set features (pd.DataFrame) and labels (pd.Series)
# features        : list of feature names

n_bootstrap = 20
scoring = "roc_auc"
rng = np.random.default_rng(42)
importance_records = []

for b in range(n_bootstrap):
    idx = rng.choice(len(X_dev), size=len(X_dev), replace=True)
    X_b, y_b = X_dev.iloc[idx], y_dev.iloc[idx]
    if len(np.unique(y_b)) < 2:
        continue
    perm = permutation_importance(
        model, X_b, y_b, n_repeats=10, random_state=42 + b, scoring=scoring,
    )
    for j, feat in enumerate(features):
        importance_records.append({"bootstrap": b, "feature": feat, "importance": perm.importances_mean[j]})

imp_df = pd.DataFrame(importance_records)
imp_summary = (
    imp_df.groupby("feature")["importance"]
    .agg(["mean", "std"])
    .assign(cv=lambda d: d["std"] / d["mean"].clip(lower=1e-6))
    .sort_values("mean", ascending=False)
)
print(f"Feature importance stability (scoring = {scoring}):")
display(imp_summary.head(15).round(4))

# Expected output:
#                    mean     std      cv
# recency_days     0.0410  0.0062  0.1512
# total_spend      0.0320  0.0058  0.1813
# tenure_months    0.0150  0.0135  0.9000

**Example commentary:**

> *"`recency_days` most important (mean AUC drop 0.041, CV 0.15). Top 3 stable (CV < 0.3). Below rank 8, CVs > 1.5."*

> *"`customer_tenure` ranks #2 but CV = 0.9. In 5 of 20 bootstraps, importance near zero. Red flag."*

> *"All 12 features have CV < 0.4 — ranking is robust. Well-specified model."*

---
# Part III — Trading strategy evaluation

## 3.1 Fama-MacBeth standard errors

`cross-sectional equity` · `factor premiums` · `anomaly research` · **Run on: train or train+dev**

**Question:** Is this cross-sectional factor premium significantly different from zero across time?

**When to use:** Cross-sectional equity factor research. Two-step procedure that handles cross-sectional correlation by averaging over time.

**Applies to:** Equity factor premiums, anomaly detection, asset pricing research.

**Theory.** Step 1: each period t, regress returns on characteristics. Collect the slope estimates {gamma_hat_t}. Step 2: test whether mean(gamma_hat) differs from zero with Newey-West SEs on the gamma series.

Cross-sectional correlations are absorbed within each period's regression. The time-series averaging tests persistence.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from collections import defaultdict

# ── Inputs ──
# panel_df    : pd.DataFrame with: date, asset_id, return, factor_1, factor_2, ...
# factors     : list of factor column names
# date_col    : str, date column name

gamma_series = defaultdict(list)
dates = sorted(panel_df[date_col].unique())

for t in dates:
    cs = panel_df[panel_df[date_col] == t].dropna()
    if len(cs) < 30:
        continue
    y = cs["return"]
    X = sm.add_constant(cs[factors])
    ols = sm.OLS(y, X).fit()
    for factor in factors:
        gamma_series[factor].append(ols.params[factor])

fm_rows = []
for factor in factors:
    gammas = np.array(gamma_series[factor])
    T = len(gammas)
    nw = sm.OLS(gammas, np.ones(T)).fit(cov_type="HAC", cov_kwds={"maxlags": int(T ** 0.25)})
    fm_rows.append({
        "factor": factor, "mean_gamma": nw.params[0], "se_nw": nw.bse[0],
        "t_stat": nw.tvalues[0], "p_value": nw.pvalues[0], "n_periods": T,
    })
display(pd.DataFrame(fm_rows).round(4))

# Expected output:
#           factor  mean_gamma   se_nw  t_stat  p_value  n_periods
# 0  book_to_market      0.0035  0.0023   1.540   0.1268        100
# 1        momentum      0.0082  0.0024   3.410   0.0009        100

**Example commentary:**

> *"Momentum: mean gamma 0.82%/month (t = 3.41, NW-corrected). Significant. Value: 0.35% (t = 1.54). Not significant — present but fragile."*

> *"All three premiums significant (t > 2.5). FM SEs are 1.4-1.7x larger than naive — moderate serial correlation in gamma."*

> *"Quality factor: t = 2.1 before NW, 1.3 after. Significance was an artifact of autocorrelation."*

## 3.2 Sharpe ratio t-test (with HAC)

`trading` · `alpha evaluation` · **Run on: OOS**

**Question:** Is the strategy's OOS Sharpe ratio significantly different from zero?

**When to use:** Any strategy backtest. With IID returns, t = SR * sqrt(T). With autocorrelated returns, use HAC. A "significant" Sharpe can still be uninteresting after costs.

**Applies to:** Any OOS strategy evaluation.

**Theory.** For IID returns: t = (mean / std) * sqrt(T) = SR * sqrt(T). Annualised Sharpe: SR_ann = SR_period * sqrt(periods_per_year). For non-IID returns (trend-following, mean-reversion), use Newey-West on the return series directly.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# ── Inputs ──
# returns       : pd.Series of OOS strategy returns (daily)
# periods_year  : int (252 daily, 12 monthly)

periods_year = 252
T = len(returns)
max_lags = int(T ** 0.25)

mu = returns.mean()
sigma = returns.std()
sharpe_annual = (mu / sigma) * np.sqrt(periods_year)

t_iid = mu / (sigma / np.sqrt(T))
nw = sm.OLS(returns.values, np.ones(T)).fit(cov_type="HAC", cov_kwds={"maxlags": max_lags})
t_hac = nw.tvalues[0]
p_hac = nw.pvalues[0]

print(f"OOS: {T} obs ({T / periods_year:.1f} years)")
print(f"Annualised Sharpe: {sharpe_annual:.2f}")
print(f"t (IID): {t_iid:.2f}, t (HAC): {t_hac:.2f}, p (HAC): {p_hac:.4f}")

# Expected output:
# OOS: 756 obs (3.0 years)
# Annualised Sharpe: 1.42
# t (IID): 5.51, t (HAC): 4.82, p (HAC): 0.0000

**Example commentary:**

> *"OOS Sharpe 1.42 over 3 years daily. HAC t = 4.8 (p < 0.001). Significant."*

> *"Sharpe 0.65 on 2 years monthly (T=24). HAC t = 1.6 (p = 0.11). Not significant — sample too short."*

> *"IID t = 3.1 vs HAC t = 1.8. The 1.7x deflation reflects return autocorrelation (trend-following). Marginally significant (p = 0.07)."*

## 3.3 Sharpe ratio decay (IS vs OOS)

`trading` · `backtest validation` · **Run on: IS + OOS**

**Question:** Does the Sharpe degrade significantly IS to OOS, signaling overfitting?

**When to use:** Always. If OOS < 50% of IS, overfitting likely. Expected deflation depends on number of trials.

**Applies to:** Any backtested strategy.

**Theory.** Bailey & Lopez de Prado (2014): when picking the best of K strategies, the expected maximum Sharpe under null (no alpha) is approximately sqrt(2 * log(K)). If the IS Sharpe barely exceeds this, performance is explainable by the search. Bootstrap both IS and OOS to compare distributions.

In [ ]:
import numpy as np
import pandas as pd

# ── Inputs ──
# returns_is, returns_oos : pd.Series of strategy returns
# n_trials     : int, number of strategies tested during IS
# periods_year : int (252 daily)

periods_year = 252
B = 2_000
rng = np.random.default_rng(42)

def ann_sharpe(r):
    return r.mean() / r.std() * np.sqrt(periods_year)

sr_is = ann_sharpe(returns_is)
sr_oos = ann_sharpe(returns_oos)
decay = 1 - sr_oos / sr_is if sr_is != 0 else np.nan

def boot_sharpe(returns, B):
    n = len(returns)
    v = returns.values
    return np.array([ann_sharpe(pd.Series(v[rng.choice(n, n, replace=True)])) for _ in range(B)])

ci_is = np.percentile(boot_sharpe(returns_is, B), [2.5, 97.5])
ci_oos = np.percentile(boot_sharpe(returns_oos, B), [2.5, 97.5])
sr0 = np.sqrt(2 * np.log(n_trials))

print(f"IS Sharpe:  {sr_is:.2f}  CI: [{ci_is[0]:.2f}, {ci_is[1]:.2f}]")
print(f"OOS Sharpe: {sr_oos:.2f}  CI: [{ci_oos[0]:.2f}, {ci_oos[1]:.2f}]")
print(f"Decay: {decay:.0%}")
print(f"Null max Sharpe (K={n_trials}): ~{sr0:.2f}")
print(f"OOS CI below IS point? {'Yes -> decay' if ci_oos[1] < sr_is else 'No'}")

# Expected output:
# IS Sharpe:  2.10  CI: [1.72, 2.48]
# OOS Sharpe: 0.90  CI: [0.41, 1.39]
# Decay: 57%
# Null max Sharpe (K=200): ~3.26
# OOS CI below IS point? Yes -> decay

**Example commentary:**

> *"IS 2.1, OOS 0.9. 57% decay. With K=200, null max ~ 3.3. IS barely exceeds what luck would produce. OOS is the honest number."*

> *"IS 1.3, OOS 1.1. 15% decay. OOS CI overlaps IS estimate. Minimal decay — signal generalises."*

> *"IS 3.5 from 5,000 trials. Null max ~ 4.1. IS below null expectation. OOS 0.4 confirms: fitted to noise."*

## 3.4 GRS test (joint alpha significance)

`factor models` · `anomaly portfolios` · `asset pricing` · **Run on: OOS**

**Question:** Are the alphas of a set of portfolios jointly different from zero?

**When to use:** Testing whether a factor model fully explains portfolio returns. GRS rejects = unexplained alpha remains.

**Applies to:** Asset pricing tests, factor model adequacy, anomaly portfolios.

**Theory.** Run N time-series regressions: R_{i,t} = alpha_i + beta_i' F_t + eps. GRS stat = [(T-N-K)/N] * [alpha' Sigma^{-1} alpha] / [1 + mu_F' Omega_F^{-1} mu_F], distributed F(N, T-N-K). Sigma = residual covariance, Omega_F = factor covariance.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

# ── Inputs ──
# portfolio_returns : pd.DataFrame, T x N excess returns
# factor_returns    : pd.DataFrame, T x K factor excess returns

T, N = portfolio_returns.shape
K = factor_returns.shape[1]

alphas = []
residuals = pd.DataFrame(index=portfolio_returns.index)
for port in portfolio_returns.columns:
    ols = sm.OLS(portfolio_returns[port], sm.add_constant(factor_returns)).fit()
    alphas.append(ols.params["const"])
    residuals[port] = ols.resid

alphas = np.array(alphas)
Sigma_inv = np.linalg.inv(residuals.cov().values)
Omega_inv = np.linalg.inv(factor_returns.cov().values)
mu_F = factor_returns.mean().values

grs = ((T - N - K) / N) * (alphas @ Sigma_inv @ alphas) / (1 + mu_F @ Omega_inv @ mu_F)
p = 1 - stats.f.cdf(grs, N, T - N - K)

print(f"GRS F-stat: {grs:.3f}, p = {p:.4f}")
print(f"Portfolios: {N}, Factors: {K}, Periods: {T}")
print(f"Mean |alpha|: {np.abs(alphas).mean():.4f}")

# Expected output:
# GRS F-stat: 2.870, p = 0.0012
# Portfolios: 25, Factors: 3, Periods: 360
# Mean |alpha|: 0.0018

**Example commentary:**

> *"GRS = 2.87 (p = 0.001) on 25 portfolios vs FF3. Significant unexplained alpha. Largest: small-growth +0.38%/month."*

> *"GRS does not reject (p = 0.23) for 10 industry portfolios vs FF5. Model adequate."*

> *"GRS rejects (p = 0.04) vs FF3, but adding UMD brings p to 0.31. Alpha was momentum."*

## 3.5 White's Reality Check / Hansen's SPA

`strategy selection` · `backtest overfitting` · **Run on: OOS / backtest**

**Question:** After trying K strategies, is the best one significantly better than the benchmark after accounting for the search?

**When to use:** Strategy selection from a large backtest universe.

**Applies to:** Strategy selection, signal zoo screening.

**Theory.** White (2000): H0 = best strategy has zero expected excess return. Bootstrap the max test stat across all K under the null. Hansen (2005) sharpens this by removing clearly inferior strategies before computing null. Both use stationary bootstrap (block length ~ T^{1/3}) to preserve dependence.

In [ ]:
import numpy as np
import pandas as pd

# ── Inputs ──
# excess_returns  : pd.DataFrame, T x K, each col = strategy - benchmark return

B = 1_000
T, K = excess_returns.shape
rng = np.random.default_rng(42)
block_length = int(np.ceil(T ** (1/3)))

mean_excess = excess_returns.mean()
se_excess = excess_returns.std() / np.sqrt(T)
t_stats = mean_excess / se_excess
t_max_obs = t_stats.max()

def stat_boot_idx(T, bl, rng):
    idx = []
    i = rng.integers(0, T)
    while len(idx) < T:
        idx.append(i % T)
        if rng.random() < 1 / bl: i = rng.integers(0, T)
        else: i += 1
    return np.array(idx[:T])

null_max = []
for _ in range(B):
    bi = stat_boot_idx(T, block_length, rng)
    bd = excess_returns.values[bi] - mean_excess.values
    bm = bd.mean(axis=0)
    bs = bd.std(axis=0) / np.sqrt(T)
    null_max.append((bm / np.where(bs > 0, bs, 1e-10)).max())

p_spa = (np.array(null_max) >= t_max_obs).mean()

print(f"Strategies: {K}, Best: {t_stats.idxmax()} (t = {t_max_obs:.2f})")
print(f"SPA p-value: {p_spa:.4f}")

# Expected output:
# Strategies: 500, Best: S42 (t = 3.41)
# SPA p-value: 0.0230

**Example commentary:**

> *"Best of 500 has t = 3.4. SPA p = 0.02 — significant after correcting for the search."*

> *"Best of 200 has t = 2.8. SPA p = 0.31. Explained by data snooping."*

> *"SPA p = 0.04 but Sharpe decays 60% OOS (3.3). Passes SPA but fragile — reduce sizing."*

## 3.6 Walk-forward / expanding window validation

`time-series` · `trading` · `any temporal ML` · **Run on: replaces single holdout**

**Question:** Is the model's performance stable across time?

**When to use:** Any temporal data. Gives a distribution of per-window metrics. Standard in quant finance. Detects decay, regime changes, drift.

**Applies to:** Return prediction, signal evaluation, any ML on time-series.

**Theory.** Train on [0, t], predict [t+1, t+h], slide forward, collect per-window metrics. Test the series: (a) t-test on mean vs baseline, (b) trend test (regress metric on window index), (c) first-half vs second-half. Expanding (growing train) when more data helps. Rolling (fixed window) when non-stationarity suspected.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score

# ── Inputs ──
# df           : pd.DataFrame sorted by date
# model_class  : sklearn estimator class
# model_params : dict
# features, target, date_col : str
# step_size    : int, OOS window size
# min_train    : int, minimum training periods

window_metrics = []
dates = sorted(df[date_col].unique())

for t in range(min_train, len(dates) - step_size, step_size):
    train = df[df[date_col].isin(dates[:t])]
    test = df[df[date_col].isin(dates[t:t + step_size])]
    model = model_class(**model_params)
    model.fit(train[features], train[target])
    pred = model.predict(test[features])
    window_metrics.append({
        "window_end": dates[t + step_size - 1], "train_n": len(train),
        "mae": mean_absolute_error(test[target], pred), "r2": r2_score(test[target], pred),
    })

wf_df = pd.DataFrame(window_metrics)
wf_df["idx"] = range(len(wf_df))
trend_p = sm.OLS(wf_df["mae"], sm.add_constant(wf_df["idx"])).fit().pvalues.iloc[1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(wf_df["idx"], wf_df["mae"], "o-", markersize=4)
axes[0].set_title("MAE by window")
axes[1].plot(wf_df["idx"], wf_df["r2"], "o-", markersize=4)
axes[1].set_title("R-squared by window")
plt.tight_layout()
plt.show()

print(f"Windows: {len(wf_df)}, MAE mean: {wf_df['mae'].mean():.2f}, std: {wf_df['mae'].std():.2f}")
print(f"Trend p: {trend_p:.4f} ({'degrading' if trend_p < 0.05 else 'stable'})")

# Expected output:
# [Plots: flat MAE and R-squared across 12 windows]
# Windows: 12, MAE mean: 580.42, std: 45.31
# Trend p: 0.6200 (stable)

**Example commentary:**

> *"12 windows, mean MAE 580 (std 45). No trend (p = 0.62). Stable performance."*

> *"MAE increases from 200 to 380 across windows. Trend p = 0.003. Signal decaying — retrain or investigate regime change."*

> *"Walk-forward Sharpes: 1.8, 1.2, 0.3, -0.1, 0.5, 0.8. Mean 0.75, std 0.7. Works in some regimes, fails in others."*

## 3.7 Combinatorial purged cross-validation (CPCV)

`return prediction` · `alpha research` · `overlapping labels` · **Run on: replaces standard CV**

**Question:** How do I cross-validate without leaking through overlapping return windows?

**When to use:** When standard k-fold leaks via overlapping labels. CPCV purges and embargoes contaminated observations.

**Applies to:** Return prediction, signal construction, any forward-looking overlapping labels.

**Theory (de Prado 2018).** Three mechanisms: (1) **Purging** — remove train obs whose label period overlaps with test. (2) **Embargo** — remove E additional train obs after test boundary (handles feature autocorrelation). (3) **Combinatorial paths** — use all C(K,2) fold pairs as test, not just K single folds.

Standard k-fold on financial data is almost always wrong. Purge+embargo minimum; CPCV is gold standard.

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

# ── Inputs ──
# df            : pd.DataFrame sorted by date, with features and target
# features      : list, target : str
# label_horizon : int, forward window in rows (e.g. 5)
# embargo_pct   : float (e.g. 0.01)
# n_folds       : int (e.g. 5)

T = len(df)
embargo_size = max(1, int(T * embargo_pct))
fold_size = T // n_folds
df = df.copy()
df["fold"] = [min(i // fold_size, n_folds - 1) for i in range(T)]

cpcv_metrics = []
for test_folds in combinations(range(n_folds), 2):
    test_mask = df["fold"].isin(test_folds)
    test_idx = df.index[test_mask]
    train_mask = ~test_mask.copy()

    for idx in test_idx:
        lo, hi = max(0, idx - label_horizon), min(T - 1, idx + label_horizon)
        train_mask[(df.index >= lo) & (df.index <= hi) & train_mask] = False

    te = test_idx.max()
    train_mask[(df.index >= te + 1) & (df.index <= te + embargo_size)] = False

    train, test = df[train_mask], df[test_mask]
    if len(train) < 50 or len(test) < 10:
        continue

    m = HistGradientBoostingRegressor(random_state=42)
    m.fit(train[features], train[target])
    p = m.predict(test[features])
    cpcv_metrics.append({
        "folds": str(test_folds), "train_n": len(train), "test_n": len(test),
        "mae": mean_absolute_error(test[target], p), "r2": r2_score(test[target], p),
    })

cpcv_df = pd.DataFrame(cpcv_metrics)
print(f"CPCV paths: {len(cpcv_df)}, horizon: {label_horizon}, embargo: {embargo_size}")
print(f"MAE: {cpcv_df['mae'].mean():.4f} +/- {cpcv_df['mae'].std():.4f}")
print(f"R2:  {cpcv_df['r2'].mean():.4f} +/- {cpcv_df['r2'].std():.4f}")
display(cpcv_df)

# Expected output:
# CPCV paths: 10, horizon: 5, embargo: 10
# MAE: 0.0142 +/- 0.0023
# R2:  0.0310 +/- 0.0180

**Example commentary:**

> *"CPCV mean R-squared 0.03 (std 0.02) with 5-day purge. Standard CV gave 0.08 — the 5pp gap was leakage."*

> *"CPCV Sharpe 0.9 (std 0.3) across 10 paths. Consistent with walk-forward. Signal generalises."*

> *"Standard CV: R-squared 0.12. CPCV: 0.01. The 11pp gap is leakage from overlapping 20-day labels."*

## 3.8 Bonferroni correction (trading context)

`trading` · `strategy selection` · `multiple testing` · **Run on: OOS**

**Question:** After screening K signals/strategies, can I trust the best one?

**When to use:** Multiple signals or parameter sets tested. For large K (>50), White's SPA (3.5) is more powerful. Bonferroni for small K.

**Applies to:** Signal screening, strategy selection, factor zoo filtering.

**Theory.** Same as 1.9/2.6. Threshold = alpha / K. Useful reframing: minimum Sharpe for significance at alpha/K over T periods is ~ z_{alpha/K} / sqrt(T). E.g. K=100, T=756 daily, alpha=0.05: z_{0.0005} = 3.29, SR_min ~ 3.29/sqrt(756) * sqrt(252) ~ 1.9 annualised.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# ── Inputs ──
# strategy_results : pd.DataFrame with: strategy_name, sharpe_annual, t_stat, p_value
# alpha            : float (default 0.05)
# periods_year     : int (252 daily)

alpha = 0.05
K = len(strategy_results)
T = periods_year * 3  # example: 3 years
bonf_threshold = alpha / K
z_bonf = stats.norm.ppf(1 - bonf_threshold / 2)
sr_min = z_bonf / np.sqrt(T) * np.sqrt(periods_year)

strategy_results = strategy_results.sort_values("sharpe_annual", ascending=False).copy()
strategy_results["sig_raw"] = strategy_results["p_value"] < alpha
strategy_results["sig_bonf"] = strategy_results["p_value"] < bonf_threshold

print(f"K={K}, threshold={bonf_threshold:.5f}, min Sharpe={sr_min:.2f}")
print(f"Raw sig: {strategy_results['sig_raw'].sum()}, Bonf sig: {strategy_results['sig_bonf'].sum()}")
display(strategy_results.head(10))

# Expected output:
# K=100, threshold=0.00050, min Sharpe=1.90
# Raw sig: 12, Bonf sig: 2

**Example commentary:**

> *"100 signals, threshold 0.0005 (~ annualised Sharpe > 1.9). Only 2 of 12 survive: momentum and value."*

> *"20 variants, best Sharpe 1.8 (p = 0.001). Bonferroni threshold 0.0025. Passes."*

> *"3 strategies. Threshold 0.017 — barely stricter than raw alpha. Minimal impact."*